# Session 12 — Distributed Training I
## 32 Virtual GPUs: Data Parallel → ZeRO-1 → ZeRO-2 → ZeRO-3

**Goal:** simulate 32 logical ranks, implement the ownership semantics of ZeRO-1/2/3, prove optimizer correctness, and quantify memory/communication trade-offs.

![Project overview](assets/00_project_cover.svg)


## 1. Mental model

Data Parallel gives every rank the same model but different data. The local gradients therefore differ, and a collective must combine them before the optimizer update. **ZeRO does not divide the neural-network compute like tensor parallelism; it removes duplicated training state.**

The mixed-precision Adam accounting used in this lab is **16 bytes/parameter**: 2 B parameter + 2 B gradient + 4 B FP32 master parameter + 4 B Adam m + 4 B Adam v.

![Experiment architecture](assets/01_experiment_architecture.svg)


In [ ]:
from pathlib import Path
import sys, subprocess
import pandas as pd
import torch
import matplotlib.pyplot as plt

repo = Path.cwd()
sys.path.insert(0, str(repo))
from src.zero_sim import *
from src.visuals import *

WORLD_SIZE = 32
LOCAL_BATCH = 4
MODEL_PARAMS = 30_000_000_000
CARD_GIB = 74.5
P_GB = 60.0
BANDWIDTH_GBPS = 50.0
shape = ModelShape(input_dim=16, hidden_dim=32, classes=4)
initial_params = init_parameters(shape, seed=7)
rank_batches = make_dataset(WORLD_SIZE, LOCAL_BATCH, shape, seed=1234)
print('world size:', WORLD_SIZE, '| global batch:', WORLD_SIZE * LOCAL_BATCH, '| toy params:', shape.num_parameters)


## 2. 32 virtual GPUs / ranks

Each logical rank gets a different local mini-batch. I back the independent gradient work with **32 CPU worker threads**, then compare it with a deterministic sequential reference.

![32 virtual ranks](assets/01_virtual_cluster.svg)


In [ ]:
loss_seq, grad_seq = all_local_gradients(initial_params, rank_batches, shape)
loss_thr, grad_thr = all_local_gradients_threaded(initial_params, rank_batches, shape, max_workers=32)
thread_grad_diff = max((a-b).abs().max().item() for a,b in zip(grad_seq, grad_thr))
print('max threaded gradient difference vs sequential:', thread_grad_diff)
assert thread_grad_diff < 1e-7
print('local gradient norm std:', torch.tensor([g.norm().item() for g in grad_seq]).std().item())


## 3. Collective correctness

The identity behind Data Parallel and ZeRO-2 is:

**All-Reduce = Reduce-Scatter + All-Gather**

![Collective flow](assets/02_collective_flow.svg)


In [ ]:
all_reduce = average_gradients(grad_seq)
rs_then_ag = cat_shards(reduce_scatter_average(grad_seq, WORLD_SIZE))
collective_diff = (all_reduce-rs_then_ag).abs().max().item()
print('max collective difference:', collective_diff)
assert collective_diff < 1e-7


## 4. Break synchronization on purpose

To prove synchronization is part of the mathematics rather than optional plumbing, I let each replica apply only its local gradient for one Adam step. The replicas should diverge.


In [ ]:
unsync_divergence, _ = unsynchronized_step_divergence(initial_params, grad_seq, lr=1e-2)
print('max rank-to-rank divergence after one unsynchronized step:', unsync_divergence)
assert unsync_divergence > 1e-6


## 5. DP / ZeRO-1 / ZeRO-2 / ZeRO-3 optimizer equivalence

All strategies start from the same parameters, use the same 32 local batches, and use the same Adam equations. Only ownership and collectives change.

![State ownership](assets/02_state_ownership.svg)


In [ ]:
_, _, _, sims, history, max_diffs = run_equivalence_demo(world_size=WORLD_SIZE, local_batch=LOCAL_BATCH, steps=5, lr=1e-2, seed=7)
history_df = pd.DataFrame(history)
display(history_df)
display(pd.DataFrame({'strategy': list(max_diffs), 'max_parameter_diff_vs_DP': list(max_diffs.values())}))
assert max(max_diffs.values()) < 1e-6


![Equivalent Adam trajectory](assets/08_correctness_trajectory.svg)

**Result:** ownership changes; the optimizer mathematics does not.


## 6. Memory accounting at world size 32

At 32 ranks, the analytical bytes/parameter/rank are:

- Data Parallel = `16`
- ZeRO-1 = `4 + 12/32 = 4.375`
- ZeRO-2 = `2 + 14/32 = 2.4375`
- ZeRO-3 = `16/32 = 0.5`

![Memory composition](assets/04_memory_composition.svg)


In [ ]:
summary = pd.DataFrame(theoretical_summary(MODEL_PARAMS, WORLD_SIZE))
summary['memory_GiB_per_rank'] = summary['memory_bytes_per_rank']/(1024**3)
summary['cluster_redundancy_x'] = [cluster_redundancy_factor(k, WORLD_SIZE) for k in ['dp','zero1','zero2','zero3']]
display(summary[['scheme','bytes_per_parameter_per_rank','memory_GiB_per_rank','communication_x_P_per_step','cluster_redundancy_x']])


## 7. The 30B memory wall

![30B fit matrix](assets/05_fit_matrix.svg)

State-only fit boundary on a 74.5 GiB GPU:

- Data Parallel: never fits
- ZeRO-1: never fits
- ZeRO-2: first fits at **32 GPUs**
- ZeRO-3: first fits at **8 GPUs**


In [ ]:
for stage in ['dp','zero1','zero2','zero3']:
    print(stage, 'first fitting world size =', first_fitting_world_size(stage, MODEL_PARAMS, CARD_GIB))
assert first_fitting_world_size('zero2', MODEL_PARAMS, CARD_GIB) == 32
assert first_fitting_world_size('zero3', MODEL_PARAMS, CARD_GIB) == 8


## 8. Memory saving versus communication

For a 30B model, one fp16 parameter copy is `P = 60 GB`. In the Session 12 model, Data Parallel / ZeRO-1 / ZeRO-2 communicate about **2P** per step, while ZeRO-3 communicates about **3P**.

![Communication pressure](assets/06_communication_pressure.svg)

![Decision map](assets/07_decision_map.svg)


In [ ]:
comm = pd.DataFrame([
    {'stage':'ZeRO-2','volume_x_P':2,'GB':120,'ideal_s_at_50GBps':communication_seconds('zero2',P_GB,BANDWIDTH_GBPS)},
    {'stage':'ZeRO-3','volume_x_P':3,'GB':180,'ideal_s_at_50GBps':communication_seconds('zero3',P_GB,BANDWIDTH_GBPS)},
])
display(comm)
print('ZeRO-2 @32 state GiB:', state_gib_per_rank('zero2',32,MODEL_PARAMS))
print('ZeRO-3 @8 state GiB:', state_gib_per_rank('zero3',8,MODEL_PARAMS))


## 9. What I understood

I remember the stages as three redundancy questions:

1. **ZeRO-1:** why should every rank keep identical optimizer history?
2. **ZeRO-2:** why should every rank retain the complete averaged gradient after synchronization?
3. **ZeRO-3:** why should every rank keep the entire parameter set resident between operations?

The deeper lesson is that **memory, computation and communication are separate axes**. ZeRO mainly attacks redundant state; it also removes duplicated optimizer work, but it does not magically divide the forward/backward neural-network computation.

> **My engineering rule:** use the lowest ZeRO stage that gives enough real memory headroom, then measure communication overlap and actual step time on the target hardware.


## 10. Scope and limitations

This is an educational state-and-collective simulator, not a DeepSpeed/FSDP performance benchmark. The toy ZeRO-3 path materializes the tiny model transiently for clarity; production systems gather layer-by-layer. Activation memory is separate from the 16 B/parameter persistent-state model, and communication timing here is volume ÷ bandwidth rather than a full NCCL topology model.


In [ ]:
test_run = subprocess.run([sys.executable,'-m','pytest','-q'], cwd=repo, capture_output=True, text=True)
print(test_run.stdout)
assert test_run.returncode == 0
checks = {
    '32 ranks': WORLD_SIZE == 32,
    'threaded gradients match': thread_grad_diff < 1e-7,
    'collective identity': collective_diff < 1e-7,
    'broken sync diverges': unsync_divergence > 1e-6,
    'ZeRO-1 matches DP': max_diffs['ZeRO-1'] < 1e-6,
    'ZeRO-2 matches DP': max_diffs['ZeRO-2'] < 1e-6,
    'ZeRO-3 matches DP': max_diffs['ZeRO-3'] < 1e-6,
    'ZeRO-2 30B boundary': first_fitting_world_size('zero2',MODEL_PARAMS,CARD_GIB) == 32,
    'ZeRO-3 30B boundary': first_fitting_world_size('zero3',MODEL_PARAMS,CARD_GIB) == 8,
}
display(pd.DataFrame({'check':checks.keys(),'passed':checks.values()}))
assert all(checks.values())
print('ALL SUBMISSION CHECKS PASSED')
